# Melbourne Housing Data Analysis

This notebook performs comprehensive data preprocessing and exploratory data analysis on Melbourne housing data from three suburbs: Highton, Ballarat, and Werribee.

## Analysis Pipeline:
1. **Data Loading & Cleaning** - Import and examine dataset structure
2. **Date Processing** - Clean sold_date format and sort chronologically  
3. **Feature Engineering** - One-hot encode categorical variables
4. **Missing Value Handling** - Impute missing values appropriately
5. **Standardization** - Normalize numerical features for modeling
6. **Visualization** - Price distributions, correlations, and trends
7. **Outlier Analysis** - Detect and visualize price outliers
8. **Temporal Analysis** - Price trends over time across suburbs

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")

In [40]:
df = pd.read_csv('melbourne_housing_data.csv')

print(f"Dataset shape: {df.shape}")
print("\nDataset overview:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

Dataset shape: (300, 10)

Dataset overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   suburb         300 non-null    object 
 1   address        300 non-null    object 
 2   property_type  300 non-null    object 
 3   bedrooms       300 non-null    int64  
 4   bathrooms      300 non-null    int64  
 5   parking        299 non-null    float64
 6   land_size      242 non-null    float64
 7   sold_price     300 non-null    int64  
 8   sold_date      300 non-null    object 
 9   listing_url    300 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 23.6+ KB
None

First 5 rows:
    suburb                        address property_type  bedrooms  bathrooms  \
0  Highton      9 Thwaites Close, Highton         House         6          2   
1  Highton     89 Thornhill Road, Highton         House         3          2   


In [44]:


def parse_date(date_str):
    return pd.to_datetime(date_str)

# Parse dates and replace original column
df['sold_date'] = df['sold_date'].apply(parse_date)

# Sort by date (newest first) - fixed parameter name
df = df.sort_values('sold_date', ascending=False, na_position='last')

In [46]:
# One-hot encode categorical variables
categorical_cols = ['suburb', 'property_type']
df_processed = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols, dummy_na=False)

print(f"Shape after encoding: {df_processed.shape}")
print(f"New columns: {df_processed.shape[1] - df.shape[1]}")

Shape after encoding: (300, 15)
New columns: 5


In [43]:
# Handle missing values
df_processed['parking'] = df_processed['parking'].fillna(0)

# Standardize numerical features
numerical_features = ['bedrooms', 'bathrooms', 'parking', 'land_size']
available_numerical = [col for col in numerical_features if col in df_processed.columns]

df_scaled = df_processed.copy()
scaler = StandardScaler()
df_scaled[available_numerical] = scaler.fit_transform(df_processed[available_numerical])

dropped_features = ['sold_price', 'sold_date', 'address', 'listing_url']
# Prepare feature matrix and target
feature_cols = [col for col in df_scaled.columns if col not in dropped_features]
X = df_scaled[feature_cols]
y = df_scaled['sold_price']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

Features (X): (300, 11)
Target (y): (300,)
